<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Transformation — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import map_coordinates

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
def locate_lab_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the image-transformation lab root."
    )
LAB_DIR = locate_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

assert DATA_FILES, f"No supported images found in: {DATA_DIR}"
def select_input_image(*keywords: str) -> Path:
    keywords = tuple(keyword.lower() for keyword in keywords)
    matches = [
        path for path in DATA_FILES
        if all(keyword in path.stem.lower() for keyword in keywords)
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching {keywords} found in {DATA_DIR}"
        )
    return matches[0]
ROLE_PATTERNS = {
    "ascent": ("ascent",),
    "ballons": ("ballon",),
    "einstein": ("einstein",),
    "tower": ("tower",),
    "peppers": ("pepper",),
}

IMAGE_FILES = {
    role: select_input_image(*patterns)
    for role, patterns in ROLE_PATTERNS.items()
}

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images discovered:", len(DATA_FILES))
print("Experiment roles :", len(IMAGE_FILES))

## 2. Load and Inspect the Reference Images


In [ ]:
rgb_images = {
    name: np.asarray(Image.open(path).convert("RGB"))
    for name, path in IMAGE_FILES.items()
}
grayscale_images = {
    name: np.asarray(Image.open(path).convert("L"))
    for name, path in IMAGE_FILES.items()
}

for name in IMAGE_FILES:
    rgb = rgb_images[name]
    gray = grayscale_images[name]

    print(
        f"{name:9s} | "
        f"RGB={str(rgb.shape):16s} "
        f"gray={str(gray.shape):12s} "
        f"dtype={gray.dtype} "
        f"range=[{gray.min()}, {gray.max()}]"
    )

In [ ]:
fig, axes = plt.subplots(1, len(rgb_images), figsize=(16, 4))

for ax, (name, image) in zip(axes, rgb_images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Intensity Transformation Model


In [ ]:
ascent = grayscale_images["ascent"]

identity = ascent.copy()

assert np.array_equal(identity, ascent)

print("Identity transformation preserves every pixel:", np.array_equal(identity, ascent))

## 4. Image Negative


In [ ]:
negative = 255 - ascent

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(negative, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Negative")

axes[2].plot(np.arange(256), 255 - np.arange(256))
axes[2].set_title("Transformation: s = 255 - r")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].set_xlim(0, 255)
axes[2].set_ylim(0, 255)
axes[2].grid(alpha=0.25)

for ax in axes[:2]:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_negative.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Brightness and Contrast


In [ ]:
def apply_linear_intensity_transform(
    image: np.ndarray,
    gain: float = 1.0,
    offset: float = 0.0,
) -> np.ndarray:

    transformed = gain * image.astype(np.float32) + offset
    return np.clip(transformed, 0, 255).astype(np.uint8)
brighter = apply_linear_intensity_transform(ascent, gain=1.0, offset=50)
darker = apply_linear_intensity_transform(ascent, gain=1.0, offset=-50)
higher_contrast = apply_linear_intensity_transform(ascent, gain=1.5, offset=-64)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

examples = [
    ("Original", ascent),
    ("Brightness +50", brighter),
    ("Brightness -50", darker),
    ("Higher contrast", higher_contrast),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_brightness_contrast.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Contrast Stretching


In [ ]:
def stretch_image_contrast(image: np.ndarray) -> np.ndarray:

    image_float = image.astype(np.float32)
    r_min = image_float.min()
    r_max = image_float.max()
    if r_max == r_min:
        return np.zeros_like(image)

    stretched = (image_float - r_min) / (r_max - r_min)
    stretched *= 255.0

    return np.clip(stretched, 0, 255).astype(np.uint8)
low_contrast_image = 90 + (ascent.astype(np.float32) / 255.0) * 75
low_contrast_image = np.clip(low_contrast_image, 0, 255).astype(np.uint8)

stretched = stretch_image_contrast(low_contrast_image)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast_image, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast_image.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before stretching")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(stretched, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("Contrast stretched")
axes[1, 0].axis("off")

axes[1, 1].hist(stretched.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After stretching")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_contrast_stretching.png", dpi=300, bbox_inches="tight")
plt.show()

print("Before:", int(low_contrast_image.min()), "to", int(low_contrast_image.max()))
print("After :", int(stretched.min()), "to", int(stretched.max()))

## 7. Logarithmic Transformation


In [ ]:
def apply_log_intensity_transform(image: np.ndarray) -> np.ndarray:

    image_float = image.astype(np.float32)

    c = 255.0 / np.log1p(255.0)

    transformed = c * np.log1p(image_float)

    return np.clip(transformed, 0, 255).astype(np.uint8)
logged = apply_log_intensity_transform(ascent)

r = np.arange(256, dtype=np.float32)
log_response_curve = (255.0 / np.log1p(255.0)) * np.log1p(r)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(logged, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Log transform")
axes[1].axis("off")

axes[2].plot(r, log_response_curve)
axes[2].set_title("Log transformation curve")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_log_transform.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Gamma / Power-Law Transformation


In [ ]:
def apply_gamma_intensity_transform(image: np.ndarray, gamma: float) -> np.ndarray:

    if gamma <= 0:
        raise ValueError("gamma must be strictly positive.")

    normalized = image.astype(np.float32) / 255.0

    transformed = normalized ** gamma

    return np.clip(transformed * 255.0, 0, 255).astype(np.uint8)
gamma_values = [0.4, 0.7, 1.0, 1.5, 2.2]

fig, axes = plt.subplots(1, len(gamma_values), figsize=(16, 3.6))

for ax, gamma in zip(axes, gamma_values):
    transformed = apply_gamma_intensity_transform(ascent, gamma)
    ax.imshow(transformed, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"γ = {gamma}")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_gamma_examples.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
r = np.linspace(0, 1, 256)

fig, ax = plt.subplots(figsize=(6, 4.5))

for gamma in gamma_values:
    ax.plot(r, r ** gamma, label=f"γ={gamma}")

ax.plot(r, r, linestyle="--", label="identity")
ax.set_title("Gamma / Power-Law Curves")
ax.set_xlabel("Normalized input r")
ax.set_ylabel("Normalized output s")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_gamma_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Histogram Equalization


In [ ]:
def equalize_image_histogram(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    intensity_histogram = np.bincount(image.ravel(), minlength=256)
    intensity_probability = intensity_histogram / image.size
    cumulative_distribution = np.cumsum(intensity_probability)
    equalization_mapping = np.round(255 * cumulative_distribution).astype(np.uint8)
    equalized_image = equalization_mapping[image]

    return equalized_image, equalization_mapping
equalized_image, equalization_map = equalize_image_histogram(low_contrast_image)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast_image, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Before equalization")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast_image.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Original histogram")

axes[1, 0].imshow(equalized_image, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After equalization")
axes[1, 0].axis("off")

axes[1, 1].hist(equalized_image.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("Equalized histogram")

for ax in [axes[0, 1], axes[1, 1]]:
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_histogram_equalization.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(np.arange(256), equalization_map)
ax.set_title("Histogram Equalization Mapping")
ax.set_xlabel("Input intensity")
ax.set_ylabel("Mapped intensity")
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_equalization_mapping.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Compare the Fundamental Intensity Transformations


In [ ]:
comparison = [
    ("Original", ascent),
    ("Negative", negative),
    ("Brighter", brighter),
    ("Contrast stretch", stretch_image_contrast(ascent)),
    ("Log", logged),
    ("Gamma 0.5", apply_gamma_intensity_transform(ascent, 0.5)),
    ("Gamma 2.0", apply_gamma_intensity_transform(ascent, 2.0)),
    ("Hist. equalized", equalize_image_histogram(ascent)[0]),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (title, image) in zip(axes.ravel(), comparison):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_intensity_transform_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Geometric Transformation Model

In [ ]:
def map_points_with_scale_and_shift(
    points: np.ndarray,
    scale_x: float = 1.15,
    scale_y: float = 0.85,
    shift_x: float = 35.0,
    shift_y: float = 20.0,
) -> np.ndarray:

    points = np.asarray(points, dtype=np.float64)
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("points must have shape (N, 2).")

    x = points[:, 0]
    y = points[:, 1]
    x_prime = scale_x * x + shift_x
    y_prime = scale_y * y + shift_y

    return np.column_stack((x_prime, y_prime))
source_points = np.array(
    [
        [0.0, 0.0],
        [120.0, 0.0],
        [120.0, 80.0],
        [0.0, 80.0],
    ],
    dtype=np.float64,
)

mapped_points = map_points_with_scale_and_shift(source_points)
expected_mapped_points = np.array(
    [
        [35.0, 20.0],
        [173.0, 20.0],
        [173.0, 88.0],
        [35.0, 88.0],
    ],
    dtype=np.float64,
)

assert mapped_points.shape == source_points.shape
assert np.all(np.isfinite(mapped_points))
assert np.allclose(mapped_points, expected_mapped_points)

print("Original coordinates:")
print(source_points)

print("\nTransformed coordinates:")
print(np.round(mapped_points, 2))

fig, ax = plt.subplots(figsize=(6, 5))
closed_source_polygon = np.vstack((source_points, source_points[0]))
closed_mapped_polygon = np.vstack((mapped_points, mapped_points[0]))

ax.plot(
    closed_source_polygon[:, 0],
    closed_source_polygon[:, 1],
    "o-",
    label="Original coordinates",
)
ax.plot(
    closed_mapped_polygon[:, 0],
    closed_mapped_polygon[:, 1],
    "o-",
    label="Mapped coordinates",
)

ax.set_title("Direct Coordinate Mapping: (x, y) → (x′, y′)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_geometric_coordinate_model.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 12. Homogeneous Coordinates


In [ ]:
def transform_point_with_matrix(matrix: np.ndarray, x: float, y: float) -> np.ndarray:
    point = np.array([x, y, 1.0], dtype=np.float64)
    transformed = matrix @ point
    return transformed[:2] / transformed[2]
identity_transform = np.eye(3)

print("Identity matrix:")
print(identity_transform)
print("Point (10, 20) ->", transform_point_with_matrix(identity_transform, 10, 20))

## 13. Fundamental Geometric Transformation Matrices


In [ ]:
def build_translation_matrix(tx: float, ty: float) -> np.ndarray:
    return np.array(
        [[1.0, 0.0, tx],
         [0.0, 1.0, ty],
         [0.0, 0.0, 1.0]]
    )
def build_scaling_matrix(sx: float, sy: float) -> np.ndarray:
    return np.array(
        [[sx, 0.0, 0.0],
         [0.0, sy, 0.0],
         [0.0, 0.0, 1.0]]
    )
def build_rotation_matrix(angle_degrees: float) -> np.ndarray:
    angle = np.deg2rad(angle_degrees)
    c = np.cos(angle)
    s = np.sin(angle)

    return np.array(
        [[c, -s, 0.0],
         [s,  c, 0.0],
         [0.0, 0.0, 1.0]]
    )
def build_shear_matrix(kx: float = 0.0, ky: float = 0.0) -> np.ndarray:
    return np.array(
        [[1.0, kx, 0.0],
         [ky, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )
def build_horizontal_reflection_matrix() -> np.ndarray:
    return np.array(
        [[-1.0, 0.0, 0.0],
         [0.0, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )

## 14. Origin-Centered vs Centered Geometry


In [ ]:
def center_transform_on_image(matrix, image_shape):
    height, width = image_shape[:2]
    center_x = (width - 1) / 2
    center_y = (height - 1) / 2
    translate_to_origin = build_translation_matrix(-center_x, -center_y)
    translate_from_origin = build_translation_matrix(center_x, center_y)
    return translate_from_origin @ matrix @ translate_to_origin

## 15. Forward Mapping vs Inverse Mapping


In [ ]:
def warp_image_affine(
    image: np.ndarray,
    forward_matrix: np.ndarray,
    output_shape: tuple[int, int] | None = None,
    interpolation_order: int = 1,
    fill_value: float = 0.0,
) -> np.ndarray:

    if output_shape is None:
        output_shape = image.shape[:2]

    out_h, out_w = output_shape
    yy, xx = np.indices((out_h, out_w), dtype=np.float64)
    output_homogeneous_coordinates = np.stack(
        [xx.ravel(), yy.ravel(), np.ones(xx.size)],
        axis=0,
    )
    inverse_transform_matrix = np.linalg.inv(forward_matrix)
    input_homogeneous_coordinates = inverse_transform_matrix @ output_homogeneous_coordinates

    x_in = input_homogeneous_coordinates[0]
    y_in = input_homogeneous_coordinates[1]
    sample_coordinates = np.vstack([y_in, x_in])
    if image.ndim == 2:
        warped = map_coordinates(
            image.astype(np.float32),
            sample_coordinates,
            order=interpolation_order,
            mode="constant",
            cval=fill_value,
        ).reshape(out_h, out_w)
    elif image.ndim == 3:
        warped_channels = []
        for channel_index in range(image.shape[2]):
            sampled = map_coordinates(
                image[..., channel_index].astype(np.float32),
                sample_coordinates,
                order=interpolation_order,
                mode="constant",
                cval=fill_value,
            ).reshape(out_h, out_w)
            warped_channels.append(sampled)

        warped = np.stack(warped_channels, axis=-1)

    else:
        raise ValueError("Expected a 2-D grayscale or 3-D color image.")

    return np.clip(warped, 0, 255).astype(np.uint8)

## 16. Translation


In [ ]:
einstein = grayscale_images["einstein"]
T = build_translation_matrix(tx=70, ty=35)
translated_image = warp_image_affine(
    einstein,
    T,

interpolation_order=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(translated_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translation: tx=70, ty=35")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_translation.png", dpi=300, bbox_inches="tight")
plt.show()

## 17. Rotation


In [ ]:
R_origin = build_rotation_matrix(30)
R_center = center_transform_on_image(build_rotation_matrix(30), einstein.shape)

rotated_origin = warp_image_affine(einstein, R_origin, interpolation_order=1)
rotated_center = warp_image_affine(einstein, R_center, interpolation_order=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(rotated_origin, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Rotation about origin")

axes[2].imshow(rotated_center, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotation about image center")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_rotation_origin_center.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Scaling and Resizing


In [ ]:
S_uniform = center_transform_on_image(

    build_scaling_matrix(0.65, 0.65),
    einstein.shape,
)

S_nonuniform = center_transform_on_image(

    build_scaling_matrix(1.35, 0.65),
    einstein.shape,
)

scaled_uniform = warp_image_affine(
    einstein,
    S_uniform,
    interpolation_order=1,
)
scaled_nonuniform = warp_image_affine(
    einstein,
    S_nonuniform,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(scaled_uniform, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Uniform scale")

axes[2].imshow(scaled_nonuniform, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Non-uniform scale")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Interpolation Comparison


In [ ]:
peppers_rgb = rgb_images["peppers"]
scale_for_interpolation = center_transform_on_image(

    build_scaling_matrix(1.7, 1.7),
    peppers_rgb.shape,
)

nearest = warp_image_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=0,
)
bilinear = warp_image_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=1,
)
bicubic = warp_image_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=3,
)
h, w = peppers_rgb.shape[:2]
crop = (
    slice(h // 3, 2 * h // 3),
    slice(w // 3, 2 * w // 3),
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(peppers_rgb)
axes[0].set_title("Original")

axes[1].imshow(nearest[crop])
axes[1].set_title("Nearest")

axes[2].imshow(bilinear[crop])
axes[2].set_title("Bilinear")

axes[3].imshow(bicubic[crop])
axes[3].set_title("Bicubic")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "14_interpolation_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 20. Reflection / Flipping


In [ ]:
height, width = einstein.shape
F = build_translation_matrix(width - 1, 0) @ build_horizontal_reflection_matrix()
reflected = warp_image_affine(einstein, F, interpolation_order=0)

reflected_numpy = einstein[:, ::-1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(reflected, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Homogeneous transform")

axes[2].imshow(reflected_numpy, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("NumPy slicing")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_reflection.png", dpi=300, bbox_inches="tight")
plt.show()

print("Two reflection methods identical:", np.array_equal(reflected, reflected_numpy))

## 21. Shear


In [ ]:
shear_centered = center_transform_on_image(

    build_shear_matrix(kx=0.35, ky=0.0),
    einstein.shape,
)

sheared = warp_image_affine(
    einstein,
    shear_centered,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(sheared, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Horizontal shear")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_shear.png", dpi=300, bbox_inches="tight")
plt.show()

## 22. Composition of Transformations


In [ ]:
translate = build_translation_matrix(70, 20)

rotate = center_transform_on_image(build_rotation_matrix(25), einstein.shape)

translate_then_rotate = rotate @ translate
rotate_then_translate = translate @ rotate
image_a = warp_image_affine(
    einstein,
    translate_then_rotate,
    interpolation_order=1,
)
image_b = warp_image_affine(
    einstein,
    rotate_then_translate,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(image_a, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translate → Rotate")

axes[2].imshow(image_b, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotate → Translate")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "17_transformation_order.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 23. General Affine Transformation


In [ ]:
affine = (
    build_translation_matrix(35, -10)
    @ center_transform_on_image(build_rotation_matrix(-18), einstein.shape)
    @ center_transform_on_image(build_shear_matrix(kx=0.18), einstein.shape)
    @ center_transform_on_image(build_scaling_matrix(0.88, 1.08), einstein.shape)
)
affine_result = warp_image_affine(
    rgb_images["einstein"],
    affine,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(rgb_images["einstein"])
axes[0].set_title("Original")

axes[1].imshow(affine_result)
axes[1].set_title("Combined affine transform")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_affine_transform.png", dpi=300, bbox_inches="tight")
plt.show()

## 24. Resizing to a New Array Shape


In [ ]:
source = Image.fromarray(rgb_images["peppers"])

original_width, original_height = source.size
new_width = original_width // 2

new_height = original_height // 2
resized_nearest = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.NEAREST,
    )
)
resized_bilinear = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BILINEAR,
    )
)
resized_bicubic = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BICUBIC,
    )
)

print("Original shape :", rgb_images["peppers"].shape)
print("Resized shape  :", resized_bilinear.shape)
print(
    "Aspect ratios  :",
    round(original_width / original_height, 4),
    "→",
    round(new_width / new_height, 4),
)

## 25. Validation Checks


In [ ]:
assert np.array_equal(identity, ascent)
assert negative.dtype == np.uint8
assert negative.min() >= 0 and negative.max() <= 255
assert brighter.dtype == np.uint8
gamma_identity = apply_gamma_intensity_transform(ascent, 1.0)
assert np.array_equal(gamma_identity, ascent)
assert np.all(np.diff(equalization_map.astype(np.int16)) >= 0)
assert mapped_points.shape == source_points.shape
assert np.allclose(mapped_points, expected_mapped_points)

assert translated_image.shape == einstein.shape
assert rotated_center.shape == einstein.shape
assert affine_result.shape == rgb_images["einstein"].shape
known_point = transform_point_with_matrix(build_translation_matrix(12, -5), 10, 20)
assert np.allclose(known_point, [22, 15])
assert np.array_equal(reflected, reflected_numpy)
assert not np.isclose(np.linalg.det(build_translation_matrix(10, 20)), 0)
assert not np.isclose(np.linalg.det(build_rotation_matrix(30)), 0)
assert not np.isclose(np.linalg.det(build_scaling_matrix(0.5, 1.5)), 0)

print("All image-transformation validation checks passed.")
REQUIRED_OUTPUTS = [
    "01_reference_images.png",
    "02_negative.png",
    "03_brightness_contrast.png",
    "04_contrast_stretching.png",
    "05_log_transform.png",
    "06_gamma_examples.png",
    "07_gamma_curves.png",
    "08_histogram_equalization.png",
    "09_equalization_mapping.png",
    "10_intensity_transform_comparison.png",
    "10_geometric_coordinate_model.png",
    "11_translation.png",
    "12_rotation_origin_center.png",
    "13_scaling.png",
    "14_interpolation_comparison.png",
    "15_reflection.png",
    "16_shear.png",
    "17_transformation_order.png",
    "18_affine_transform.png",
]
missing_outputs = [
    output_name
    for output_name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / output_name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing_outputs)
    )

print(f"Output validation passed: {len(REQUIRED_OUTPUTS)} files.")
